# CatBoost — Gradient Boosting That Speaks 'Category' Fluently

---

## What Is CatBoost?

CatBoost (short for **Cat**egorical **Boost**ing) is a gradient-boosted decision-tree library developed by Yandex (the Russian Google) and open-sourced in 2017. Its two killer features:

1. **Native categorical-feature handling** — you hand CatBoost raw strings (`'Red'`, `'New York'`, `'Gold'`) and it figures out the encoding automatically using a technique called *Ordered Target Encoding*.
2. **Ordered boosting** — a clever trick that prevents a subtle but serious form of data leakage during training that afflicts all other gradient boosting libraries.

### Real-World Analogy

Imagine you're teaching a student to grade essays. With XGBoost or LightGBM, before the student can start learning you have to **translate every descriptive label** (`'Excellent'`, `'Good'`, `'Poor'`) into a number yourself, and you could easily do the translation **wrong** or lose information. CatBoost is like a student who already speaks multiple languages — you can hand it essays written in any language and it handles the translation itself, internally, in a smarter way than you could easily do manually.

---

## Why Should You Care?

Most real-world datasets are **messy mixtures of numbers and categories**: customer ID, city, product category, subscription tier, device type. The typical preprocessing pipeline for other boosting libraries requires:

```
raw data → OrdinalEncoder / OneHotEncoder → encoded matrix → XGBoost/LightGBM
```

This is error-prone and can cause **target leakage** if you encode using the full dataset before splitting. CatBoost eliminates this step entirely.

| Feature | XGBoost | LightGBM | **CatBoost** |
|---|---|---|---|
| Native categoricals | No | Partial (needs integers) | **Yes (raw strings!)** |
| Ordered boosting | No | No | **Yes** |
| Training speed | Medium | Fast | Medium-Fast |
| GPU support | Yes | Yes | Yes |
| Default parameters | Need tuning | Need tuning | **Works well out-of-the-box** |

---

## Prerequisites

- Python basics (lists, dicts, functions)
- Pandas DataFrames (what a column is)
- Basic ML concepts (train/test split, classification vs regression)
- Familiarity with XGBoost or Scikit-Learn helps but is not required

---

## Table of Contents

1. Installation & Setup
2. The Core Idea: How CatBoost Handles Categoricals
3. Ordered Boosting — The Anti-Leakage Trick
4. CatBoostClassifier — Sklearn-Compatible API
5. CatBoostRegressor — Same API for Regression
6. The Pool Object — Native Data Container
7. Key Hyperparameters Explained
8. Feature Importance & SHAP Values
9. CatBoost vs XGBoost vs LightGBM — Head-to-Head on Categorical Data
10. Mini Project — House Price Prediction with Rich Categorical Features
11. Common Pitfalls & How to Avoid Them
12. Interview Q&A
13. Resources
14. Summary & What's Next

---

**Official Docs:** https://catboost.ai/en/docs/  
**GitHub:** https://github.com/catboost/catboost  
**Original Paper:** https://arxiv.org/abs/1706.09516  
**YouTube — CatBoost explained (Yandex):** https://www.youtube.com/watch?v=s3VmuVPfu0s  
**Kaggle Tutorial:** https://www.kaggle.com/learn/machine-learning-explainability  

## 1. Installation & Setup

```bash
pip install catboost
```

CatBoost works on macOS, Linux, and Windows. GPU support is automatic if CUDA is available.

In [ ]:
catboosttry:
    from catboost import CatBoostClassifier, CatBoostRegressor, Pool
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, roc_auc_score
    import warnings; warnings.filterwarnings("ignore")
    print(f"CatBoost {__import__("catboost").__version__} ready")
except ImportError:
    raise SystemExit("Run: pip install catboost scikit-learn pandas numpy matplotlib")


## 2. The Core Idea: How CatBoost Handles Categoricals

### The Problem with Standard Approaches

Imagine you're predicting house prices and you have a column `neighborhood` with values: `'Downtown'`, `'Suburbs'`, `'Rural'`.

**Option A — Label Encoding:**
```
Downtown → 0, Rural → 1, Suburbs → 2
```
This is **wrong** — it implies Downtown < Rural < Suburbs, but there's no natural ordering!

**Option B — One-Hot Encoding:**
```
Downtown → [1, 0, 0], Rural → [0, 1, 0], Suburbs → [0, 0, 1]
```
This works but creates many columns (bad for high-cardinality features like city names with 1000+ unique values).

**Option C — Target Encoding (naive):**
```
Downtown → mean(price | neighborhood='Downtown') = 450,000
```
This uses the target to encode — **DATA LEAKAGE!** You're using the answer to calculate the input.

### CatBoost's Solution: Ordered Target Encoding

CatBoost uses **Ordered Target Encoding** — a mathematically rigorous version of target encoding that prevents leakage:

When computing the encoding for sample `i` in category `C`, CatBoost uses **only the samples that appeared BEFORE sample i** (in a random permutation of the training set):

```
encoding(sample_i, category_C) = 
    (sum of targets for all samples j < i where category_j == C) + prior
    ─────────────────────────────────────────────────────────────────
    (count of samples j < i where category_j == C) + 1
```

The `prior` is the global mean target. The key insight: **sample i's own label never influences its own encoding**, so no leakage occurs.

In [ ]:
# ==================================================
# DEMO: CatBoost accepts RAW STRING categoricals!
# No encoding needed — just pass the strings directly.
# ==================================================

# Create a tiny dataset with raw string categorical columns
data = pd.DataFrame({
    'city':        ['New York', 'Paris', 'Tokyo', 'London', 'New York', 'Paris',
                    'Tokyo', 'London', 'New York', 'Paris'],
    'product_type': ['Electronics', 'Clothing', 'Electronics', 'Food', 'Clothing',
                     'Electronics', 'Food', 'Clothing', 'Electronics', 'Food'],
    'price':       [999, 50, 1200, 5, 75, 800, 8, 30, 1100, 12],
    'purchased':   [1, 0, 1, 0, 1, 1, 0, 0, 1, 0]   # target
})

X = data[['city', 'product_type', 'price']]
y = data['purchased']

# Tell CatBoost which columns are categorical — pass column NAMES or INDICES
cat_features = ['city', 'product_type']  # strings are fine!

model = CatBoostClassifier(
    iterations=20,
    cat_features=cat_features,
    verbose=0  # suppress training log
)
model.fit(X, y)
print("Predictions:", model.predict(X).flatten())
print("Probabilities:\n", model.predict_proba(X)[:, 1].round(3))
print("\n✓ Notice: we passed raw strings like 'New York' and 'Electronics' — CatBoost handled everything!")

## 3. Ordered Boosting — The Anti-Leakage Trick

### Standard Gradient Boosting (XGBoost/LightGBM)

In regular gradient boosting:
1. Train Tree 1 on ALL training samples
2. Compute residuals (errors) for ALL training samples using Tree 1's predictions
3. Train Tree 2 on those residuals
4. Repeat...

**The subtle bug:** When computing residuals in step 2, each sample's residual is computed using a tree that was trained **including that sample itself**. The tree has already "seen" that sample, so the residuals are slightly underestimated (the model is too confident). This is called **prediction shift**.

### CatBoost's Ordered Boosting

CatBoost maintains **multiple permutations** of the training data. For each sample, it computes residuals using a model that was trained **only on samples that appeared earlier in the permutation** — the sample has never been seen by the model computing its own residual.

**Think of it like a school exam:** In regular boosting, the teacher corrects your test using an answer key they wrote while looking at your answers (circular!). In ordered boosting, the answer key was written before the teacher saw your answers.

**Trade-off:** Ordered boosting is more correct but slower. CatBoost automatically uses **Plain boosting** (standard approach) for large datasets as a speed optimization.

## 4. CatBoostClassifier — The Sklearn-Compatible API

Works exactly like `sklearn.ensemble.GradientBoostingClassifier`: `.fit()`, `.predict()`, `.predict_proba()`, `.score()`.

In [ ]:
# ==================================================
# REALISTIC DEMO: Employee Attrition Prediction
# Mix of numeric and categorical features
# ==================================================

np.random.seed(42)
n = 1000

departments   = np.random.choice(['Engineering', 'Sales', 'HR', 'Finance', 'Marketing'], n)
job_levels    = np.random.choice(['Junior', 'Mid', 'Senior', 'Lead', 'Manager'], n)
locations     = np.random.choice(['NYC', 'SF', 'Chicago', 'Austin', 'Remote'], n)
salary_band   = np.random.choice(['Low', 'Medium', 'High', 'Very High'], n)
satisfaction  = np.random.randint(1, 6, n)            # 1-5 survey score
years_company = np.random.exponential(4, n).clip(0, 30).astype(int)
overtime_hrs  = np.random.randint(0, 20, n)

# Build attrition probability from features (realistic signal)
p_attrition = 0.15
p_attrition += (salary_band == 'Low') * 0.15
p_attrition += (satisfaction <= 2) * 0.20
p_attrition += (overtime_hrs > 12) * 0.10
p_attrition += (years_company < 2) * 0.10
p_attrition -= (job_levels == 'Manager') * 0.05
p_attrition = p_attrition.clip(0.02, 0.95)
left = (np.random.rand(n) < p_attrition).astype(int)

df = pd.DataFrame({
    'department':   departments,
    'job_level':    job_levels,
    'location':     locations,
    'salary_band':  salary_band,
    'satisfaction': satisfaction,
    'years':        years_company,
    'overtime':     overtime_hrs,
    'left':         left
})

print(f"Dataset shape: {df.shape}")
print(f"Attrition rate: {left.mean():.1%}")
print("\nSample rows:")
df.head()

In [ ]:
# Split data
X = df.drop('left', axis=1)
y = df['left']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Identify categorical columns
cat_cols = ['department', 'job_level', 'location', 'salary_band']

# Train CatBoostClassifier
clf = CatBoostClassifier(
    iterations=300,          # number of trees
    learning_rate=0.05,      # step size per tree
    depth=6,                 # max depth of each tree
    cat_features=cat_cols,   # specify categoricals by name
    eval_metric='AUC',       # metric to watch on validation set
    random_seed=42,
    verbose=0
)

# CatBoost has built-in eval set support (like XGBoost)
clf.fit(
    X_train, y_train,
    eval_set=(X_test, y_test)
)

# Evaluate
y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}")
print(f"Best iteration: {clf.best_iteration_}")

## 5. CatBoostRegressor — Same API for Regression

In [ ]:
# ==================================================
# DEMO: CatBoostRegressor on salary prediction
# ==================================================

np.random.seed(7)
n = 800

education = np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n)
field     = np.random.choice(['Engineering', 'Business', 'Arts', 'Science', 'Law'], n)
city_size = np.random.choice(['Small', 'Medium', 'Large', 'Metro'], n)
experience= np.random.randint(0, 30, n)
age       = (experience + np.random.randint(22, 30, n)).clip(22, 65)

# Salary formula
base = 35000
edu_bonus  = {'High School': 0, 'Bachelor': 15000, 'Master': 25000, 'PhD': 35000}
field_mult = {'Engineering': 1.4, 'Business': 1.2, 'Arts': 0.9, 'Science': 1.1, 'Law': 1.3}
city_mult  = {'Small': 0.85, 'Medium': 1.0, 'Large': 1.15, 'Metro': 1.35}

salary = np.array([
    (base + edu_bonus[e] + experience[i] * 1500) * field_mult[f] * city_mult[c]
    for i, (e, f, c) in enumerate(zip(education, field, city_size))
]) + np.random.normal(0, 5000, n)

df_reg = pd.DataFrame({
    'education': education,
    'field': field,
    'city_size': city_size,
    'experience': experience,
    'age': age,
    'salary': salary.clip(20000)
})

X_r = df_reg.drop('salary', axis=1)
y_r = df_reg['salary']
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

reg = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=5,
    cat_features=['education', 'field', 'city_size'],
    eval_metric='RMSE',
    random_seed=42,
    verbose=0
)
reg.fit(X_tr, y_tr, eval_set=(X_te, y_te))

rmse = np.sqrt(mean_squared_error(y_te, reg.predict(X_te)))
print(f"RMSE: ${rmse:,.0f}")
print(f"Median salary in test: ${y_te.median():,.0f}")
print(f"Relative error: {rmse/y_te.median():.1%}")

## 6. The Pool Object — CatBoost's Native Data Container

Just as XGBoost has `DMatrix` and LightGBM has `lgb.Dataset`, CatBoost has `Pool`. It pre-processes data once and stores it efficiently.

**When to use Pool:**
- Large datasets (avoid re-processing on every call)
- When you want to specify column names, weights, or baseline predictions
- Using the low-level `catboost.train()` API

In [ ]:
from catboost import Pool

# Re-use the attrition dataset from above
X_train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=cat_cols   # specify here instead of in the model
)

X_test_pool = Pool(
    data=X_test,
    label=y_test,
    cat_features=cat_cols
)

print(f"Train Pool: {X_train_pool.num_row()} rows, {X_train_pool.num_col()} cols")
print(f"Test  Pool: {X_test_pool.num_row()} rows, {X_test_pool.num_col()} cols")

# Train using Pool objects — now no need to specify cat_features in the model
clf_pool = CatBoostClassifier(
    iterations=200,
    learning_rate=0.05,
    depth=5,
    eval_metric='AUC',
    random_seed=42,
    verbose=0
)
clf_pool.fit(X_train_pool, eval_set=X_test_pool)

y_proba_pool = clf_pool.predict_proba(X_test_pool)[:, 1]
print(f"\nROC-AUC using Pool: {roc_auc_score(y_test, y_proba_pool):.4f}")

## 7. Key Hyperparameters Explained

| Parameter | Default | What It Controls | When to Change |
|---|---|---|---|
| `iterations` | 1000 | Number of trees | More trees + lower LR = better (use early stopping) |
| `learning_rate` | auto (~0.03) | Step size per tree | Lower = more trees needed but smoother |
| `depth` | 6 | Max depth of each tree | Deeper = more complex (risk of overfitting); range 4-10 |
| `l2_leaf_reg` | 3 | L2 regularization on leaf weights | Higher = simpler model, less overfit |
| `min_data_in_leaf` | 1 | Min samples in a leaf | Higher = more regularization |
| `bagging_temperature` | 1.0 | Randomness in bootstrap sampling | Higher = more random |
| `random_strength` | 1.0 | Randomness when choosing splits | Higher = more regularization |
| `border_count` | 254 | Buckets for numeric features | More = more precise but slower |
| `grow_policy` | `SymmetricTree` | Tree growth strategy | `Lossguide` = LightGBM-style leaf-wise |
| `boosting_type` | `Ordered` | Ordered vs Plain boosting | `Plain` is faster for large datasets |
| `task_type` | `CPU` | CPU or GPU training | Set `GPU` if you have a CUDA GPU |

### Most Important for Tuning

```
1. iterations + learning_rate  (always tune together; use early stopping)
2. depth                       (start at 6, try 4-10)
3. l2_leaf_reg                 (try 1, 3, 5, 10, 30, 100)
4. random_strength             (try 0.5, 1, 2)
5. bagging_temperature         (try 0.5, 1, 2)
```

In [ ]:
# ==================================================
# DEMO: Early Stopping — let CatBoost find the
# optimal number of trees automatically
# ==================================================

clf_es = CatBoostClassifier(
    iterations=1000,          # max trees (will stop early)
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    cat_features=cat_cols,
    eval_metric='AUC',
    early_stopping_rounds=50, # stop if no improvement for 50 rounds
    random_seed=42,
    verbose=100               # print every 100 iterations
)

clf_es.fit(X_train, y_train, eval_set=(X_test, y_test))

print(f"\nBest iteration   : {clf_es.best_iteration_}")
print(f"Best AUC on eval : {clf_es.best_score_['validation']['AUC']:.4f}")

## 8. Feature Importance & SHAP Values

CatBoost supports multiple feature importance methods:

| Method | What It Measures | When to Use |
|---|---|---|
| `PredictionValuesChange` | How much predictions change when a feature is removed | Quick global overview |
| `LossFunctionChange` | How much the loss changes without the feature | More accurate than PVC |
| `ShapValues` | Additive SHAP values for each prediction | Detailed local explanations |

In [ ]:
# Feature importance — PredictionValuesChange (fast)
fi = clf.get_feature_importance(prettified=True)
print("Feature Importance (PredictionValuesChange):")
print(fi.to_string(index=False))

In [ ]:
# Visualize feature importance
fig, ax = plt.subplots(figsize=(8, 5))
fi_sorted = fi.sort_values('Importances')
colors = ['#e74c3c' if c in cat_cols else '#3498db' for c in fi_sorted['Feature Id']]
ax.barh(fi_sorted['Feature Id'], fi_sorted['Importances'], color=colors)
ax.set_xlabel('Feature Importance (PredictionValuesChange)')
ax.set_title('CatBoost Feature Importance\n(red = categorical, blue = numeric)')

# Legend
from matplotlib.patches import Patch
legend = [Patch(color='#e74c3c', label='Categorical'), Patch(color='#3498db', label='Numeric')]
ax.legend(handles=legend, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Values — per-sample explanations
# shap_values[i] tells you: for sample i, how much did each feature
# push the prediction above/below the baseline?

shap_values = clf.get_feature_importance(
    data=X_test_pool,
    type='ShapValues'
)

# shap_values shape: (n_samples, n_features + 1)  <- last col is bias
print(f"SHAP values shape: {shap_values.shape}")
print(f"Feature names: {X_test.columns.tolist()}")

# Mean absolute SHAP — global importance
mean_shap = np.abs(shap_values[:, :-1]).mean(axis=0)
shap_df = pd.DataFrame({'feature': X_test.columns, 'mean_|shap|': mean_shap})
shap_df = shap_df.sort_values('mean_|shap|', ascending=False)
print("\nMean |SHAP| per feature:")
print(shap_df.to_string(index=False))

# SHAP for a single prediction — explains WHY the model said what it said
sample_idx = 0
print(f"\nSHAP explanation for sample {sample_idx}:")
print(f"  Predicted probability: {clf.predict_proba(X_test.iloc[[sample_idx]])[:, 1][0]:.3f}")
for feat, sv in zip(X_test.columns, shap_values[sample_idx, :-1]):
    direction = '↑ increases' if sv > 0 else '↓ decreases'
    print(f"  {feat:12s}: {sv:+.4f}  {direction} attrition probability")

## 9. CatBoost vs XGBoost vs LightGBM — Head-to-Head on Categorical Data

The most informative comparison is on a **categorical-heavy dataset** — where CatBoost has the biggest advantage.

In [ ]:
import time
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import OrdinalEncoder

# Larger dataset for fair comparison
np.random.seed(0)
N = 5000

industries   = np.random.choice(['Tech', 'Finance', 'Healthcare', 'Retail', 'Education',
                                  'Manufacturing', 'Government', 'Media', 'Legal', 'Transport'], N)
company_size = np.random.choice(['Startup', 'SMB', 'Mid-Market', 'Enterprise', 'Fortune500'], N)
country      = np.random.choice(['USA', 'UK', 'Germany', 'India', 'Canada', 'France',
                                  'Australia', 'Japan', 'Brazil', 'Singapore'], N)
plan         = np.random.choice(['Free', 'Basic', 'Pro', 'Enterprise'], N)
tenure_months = np.random.randint(1, 60, N)
usage_score  = np.random.randint(0, 100, N)

# Churn signal
p_churn = 0.2
p_churn += (plan == 'Free') * 0.25
p_churn += (usage_score < 30) * 0.20
p_churn += (tenure_months < 6) * 0.15
p_churn -= (company_size == 'Enterprise') * 0.10
p_churn = np.clip(p_churn, 0.02, 0.95)
churn = (np.random.rand(N) < p_churn).astype(int)

df_comp = pd.DataFrame({
    'industry': industries, 'company_size': company_size,
    'country': country, 'plan': plan,
    'tenure': tenure_months, 'usage': usage_score,
    'churn': churn
})

cat_cols_comp = ['industry', 'company_size', 'country', 'plan']
X_c = df_comp.drop('churn', axis=1)
y_c = df_comp['churn']
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42, stratify=y_c)

print(f"Dataset: {N} rows, {X_c.shape[1]} features ({len(cat_cols_comp)} categorical)")
print(f"Churn rate: {churn.mean():.1%}\n")

results = {}

# ---- 1. CatBoost (no encoding needed) ----
t0 = time.time()
cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                         cat_features=cat_cols_comp, random_seed=42, verbose=0)
cb.fit(X_tr_c, y_tr_c)
cb_time = time.time() - t0
results['CatBoost'] = {'auc': roc_auc_score(y_te_c, cb.predict_proba(X_te_c)[:, 1]),
                        'time': cb_time, 'preprocessing': 'None (native)'}

# ---- 2. XGBoost (needs ordinal encoding) ----
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_tr_xgb = X_tr_c.copy()
X_te_xgb  = X_te_c.copy()
X_tr_xgb[cat_cols_comp] = oe.fit_transform(X_tr_c[cat_cols_comp])
X_te_xgb[cat_cols_comp]  = oe.transform(X_te_c[cat_cols_comp])

t0 = time.time()
xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                     random_state=42, eval_metric='logloss', verbosity=0)
xgb.fit(X_tr_xgb, y_tr_c)
xgb_time = time.time() - t0
results['XGBoost'] = {'auc': roc_auc_score(y_te_c, xgb.predict_proba(X_te_xgb)[:, 1]),
                       'time': xgb_time, 'preprocessing': 'OrdinalEncoder'}

# ---- 3. LightGBM (needs integer-encoded categoricals) ----
X_tr_lgb = X_tr_c.copy()
X_te_lgb  = X_te_c.copy()
X_tr_lgb[cat_cols_comp] = oe.fit_transform(X_tr_c[cat_cols_comp])
X_te_lgb[cat_cols_comp]  = oe.transform(X_te_c[cat_cols_comp])
for col in cat_cols_comp:
    X_tr_lgb[col] = X_tr_lgb[col].astype(int)
    X_te_lgb[col]  = X_te_lgb[col].astype(int)

t0 = time.time()
lgb = LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                      categorical_feature=cat_cols_comp, random_state=42, verbose=-1)
lgb.fit(X_tr_lgb, y_tr_c)
lgb_time = time.time() - t0
results['LightGBM'] = {'auc': roc_auc_score(y_te_c, lgb.predict_proba(X_te_lgb)[:, 1]),
                        'time': lgb_time, 'preprocessing': 'OrdinalEncoder + int cast'}

# Print comparison
print(f"{'Model':<12} {'ROC-AUC':<12} {'Train Time':<14} {'Preprocessing'}")
print("-" * 60)
for model, r in results.items():
    print(f"{model:<12} {r['auc']:.4f}       {r['time']:.2f}s          {r['preprocessing']}")

## 10. Mini Project — Real Estate Price Prediction with Rich Categorical Features

### The Business Problem

A real estate company wants to predict house sale prices. Their database has a rich mix of numeric measurements AND categorical descriptions that would normally require complex preprocessing.

**Goal:** Build a CatBoost regressor that handles all the categorical features natively and achieves low prediction error.

**Features:**
- Numeric: square footage, lot size, year built, bedrooms, bathrooms, garage spaces
- Categorical (high cardinality): neighborhood (50 unique), building material (8 unique), roof type (6 unique), heating type (5 unique), foundation type (4 unique), house style (7 unique), condition (5 unique), sale type (5 unique)

In [ ]:
# ==================================================
# STEP 1: Generate realistic real estate dataset
# ==================================================

np.random.seed(2024)
N = 3000

# Categorical features with meaningful cardinality
neighborhoods = [f'Neighborhood_{i}' for i in range(50)]  # high cardinality!
materials     = ['Brick', 'Wood', 'Concrete', 'Steel', 'Stone', 'Stucco', 'Vinyl', 'Fiber Cement']
roof_types    = ['Gable', 'Hip', 'Flat', 'Mansard', 'Gambrel', 'Shed']
heating       = ['Gas', 'Electric', 'Oil', 'Heat Pump', 'Radiant']
foundation    = ['Poured Concrete', 'Brick', 'Block', 'Slab']
styles        = ['Ranch', 'Colonial', 'Victorian', 'Craftsman', 'Modern', 'Tudor', 'Cape Cod']
conditions    = ['Excellent', 'Good', 'Average', 'Fair', 'Poor']
sale_types    = ['Warranty Deed', 'New', 'Contract of Deed', 'Foreclosure', 'Estate']

# Assign quality scores to conditions (for price formula)
cond_quality = {'Excellent': 1.30, 'Good': 1.15, 'Average': 1.00, 'Fair': 0.85, 'Poor': 0.70}
neigh_premium = {n: np.random.uniform(0.7, 1.5) for n in neighborhoods}  # each neighborhood has a price premium
mat_premium   = dict(zip(materials, np.random.uniform(0.9, 1.2, len(materials))))

df_re = pd.DataFrame({
    'neighborhood':  np.random.choice(neighborhoods, N),
    'material':      np.random.choice(materials, N),
    'roof_type':     np.random.choice(roof_types, N),
    'heating':       np.random.choice(heating, N),
    'foundation':    np.random.choice(foundation, N),
    'style':         np.random.choice(styles, N),
    'condition':     np.random.choice(conditions, N, p=[0.1, 0.3, 0.35, 0.15, 0.10]),
    'sale_type':     np.random.choice(sale_types, N, p=[0.70, 0.15, 0.05, 0.05, 0.05]),
    'sqft':          np.random.randint(600, 5000, N),
    'lot_sqft':      np.random.randint(3000, 50000, N),
    'year_built':    np.random.randint(1900, 2023, N),
    'bedrooms':      np.random.randint(1, 7, N),
    'bathrooms':     np.round(np.random.choice([1, 1.5, 2, 2.5, 3, 3.5, 4], N, p=[0.1,0.15,0.3,0.2,0.15,0.07,0.03]), 1),
    'garage_spaces': np.random.choice([0, 1, 2, 3], N, p=[0.1, 0.3, 0.45, 0.15]),
})

# Compute price based on features (realistic formula)
prices = []
for _, row in df_re.iterrows():
    base_price = 50 * row['sqft']  # $50/sqft base
    base_price *= neigh_premium[row['neighborhood']]
    base_price *= cond_quality[row['condition']]
    base_price *= mat_premium[row['material']]
    base_price += row['lot_sqft'] * 0.5
    base_price += row['garage_spaces'] * 15000
    base_price += row['bathrooms'] * 8000
    base_price += max(0, 2023 - row['year_built']) * -300  # older = cheaper
    base_price *= np.random.uniform(0.90, 1.10)  # market noise
    prices.append(max(base_price, 50000))

df_re['sale_price'] = prices

print(f"Dataset: {df_re.shape}")
print(f"Price range: ${df_re['sale_price'].min():,.0f} – ${df_re['sale_price'].max():,.0f}")
print(f"Median price: ${df_re['sale_price'].median():,.0f}")
print(f"\nCategorical features: {[c for c in df_re.columns if df_re[c].dtype == object]}")

In [ ]:
# ==================================================
# STEP 2: Prepare data and train CatBoost Regressor
# ==================================================

X_re = df_re.drop('sale_price', axis=1)
y_re = df_re['sale_price']

X_re_tr, X_re_te, y_re_tr, y_re_te = train_test_split(
    X_re, y_re, test_size=0.2, random_state=42
)

cat_features_re = ['neighborhood', 'material', 'roof_type', 'heating',
                    'foundation', 'style', 'condition', 'sale_type']

# Create Pool objects
train_pool = Pool(X_re_tr, y_re_tr, cat_features=cat_features_re)
test_pool  = Pool(X_re_te, y_re_te, cat_features=cat_features_re)

# Train model with early stopping
reg_re = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=7,
    l2_leaf_reg=3,
    eval_metric='RMSE',
    early_stopping_rounds=50,
    random_seed=42,
    verbose=200
)

reg_re.fit(train_pool, eval_set=test_pool)

print(f"\nOptimal trees: {reg_re.best_iteration_}")

In [ ]:
# ==================================================
# STEP 3: Evaluate — multiple metrics
# ==================================================

y_pred_re = reg_re.predict(X_re_te)

rmse = np.sqrt(mean_squared_error(y_re_te, y_pred_re))
mae  = np.mean(np.abs(y_re_te - y_pred_re))
mape = np.mean(np.abs((y_re_te - y_pred_re) / y_re_te)) * 100

# R-squared
ss_res = np.sum((y_re_te - y_pred_re) ** 2)
ss_tot = np.sum((y_re_te - y_re_te.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

print("=" * 40)
print("   REAL ESTATE MODEL EVALUATION")
print("=" * 40)
print(f"RMSE : ${rmse:>12,.0f}")
print(f"MAE  : ${mae:>12,.0f}")
print(f"MAPE : {mape:>11.1f}%")
print(f"R²   : {r2:>12.4f}")
print("=" * 40)
print(f"Median sale price: ${y_re_te.median():>10,.0f}")
print(f"Error as % of median: {rmse/y_re_te.median()*100:.1f}%")

In [ ]:
# ==================================================
# STEP 4: Visualize results
# ==================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('CatBoost Real Estate Price Prediction', fontsize=14, fontweight='bold')

# Panel 1: Predicted vs Actual
ax = axes[0]
ax.scatter(y_re_te / 1e6, y_pred_re / 1e6, alpha=0.3, s=15, color='steelblue')
lims = [min(y_re_te.min(), y_pred_re.min()) / 1e6,
        max(y_re_te.max(), y_pred_re.max()) / 1e6]
ax.plot(lims, lims, 'r--', lw=2, label='Perfect prediction')
ax.set_xlabel('Actual Price ($M)')
ax.set_ylabel('Predicted Price ($M)')
ax.set_title(f'Predicted vs Actual\n(R² = {r2:.3f})')
ax.legend()

# Panel 2: Residual Distribution
ax = axes[1]
residuals = (y_pred_re - y_re_te) / 1000
ax.hist(residuals, bins=50, color='darkorange', edgecolor='white', alpha=0.8)
ax.axvline(0, color='red', linestyle='--', lw=2)
ax.set_xlabel('Residual ($K)')
ax.set_ylabel('Count')
ax.set_title('Residual Distribution')

# Panel 3: Feature Importance
ax = axes[2]
fi_re = reg_re.get_feature_importance(prettified=True).head(10)
fi_re_sorted = fi_re.sort_values('Importances')
colors_re = ['#e74c3c' if f in cat_features_re else '#3498db'
              for f in fi_re_sorted['Feature Id']]
ax.barh(fi_re_sorted['Feature Id'], fi_re_sorted['Importances'], color=colors_re)
ax.set_xlabel('Importance')
ax.set_title('Top 10 Feature Importances\n(red=categorical, blue=numeric)')

plt.tight_layout()
plt.show()

print("\nKey Insight: Despite 'neighborhood' having 50 unique values, CatBoost handled")
print("it natively via Ordered Target Encoding — no manual preprocessing required!")

In [ ]:
# ==================================================
# STEP 5: Predict on a NEW house (inference demo)
# ==================================================

new_house = pd.DataFrame([{
    'neighborhood':  'Neighborhood_42',
    'material':      'Brick',
    'roof_type':     'Hip',
    'heating':       'Gas',
    'foundation':    'Poured Concrete',
    'style':         'Colonial',
    'condition':     'Good',
    'sale_type':     'Warranty Deed',
    'sqft':          2400,
    'lot_sqft':      8500,
    'year_built':    1998,
    'bedrooms':      4,
    'bathrooms':     2.5,
    'garage_spaces': 2
}])

predicted_price = reg_re.predict(new_house)[0]
print("New House Listing:")
print("  Neighborhood: Neighborhood_42  |  Style: Colonial")
print("  2,400 sqft  |  8,500 sqft lot  |  Built: 1998")
print("  4 bed, 2.5 bath, 2-car garage  |  Condition: Good")
print(f"\nCatBoost Predicted Sale Price: ${predicted_price:,.0f}")

## 11. Common Pitfalls & How to Avoid Them

### Pitfall 1: Not Specifying `cat_features`

If you forget to tell CatBoost which columns are categorical, it treats them as numeric (usually fails or gives wrong results).

```python
# WRONG — CatBoost will try to interpret 'New York' as a float
model = CatBoostClassifier(iterations=100, verbose=0)
model.fit(X, y)  # error or garbage predictions

# CORRECT — always specify cat_features
model = CatBoostClassifier(iterations=100, cat_features=['city', 'country'], verbose=0)
model.fit(X, y)
```

### Pitfall 2: NaN in Categorical Columns

CatBoost handles NaN in **numeric** columns natively. But for **categorical** columns, NaN must be filled before training.

```python
# Fill categorical NaNs with a sentinel string
df['city'].fillna('Unknown', inplace=True)
```

### Pitfall 3: Confusing `depth` with LightGBM's `num_leaves`

CatBoost uses **symmetric trees** by default (all branches at each depth level split on the same feature). This is very different from LightGBM's leaf-wise growth. A depth of 6 in CatBoost = 64 leaves but all leaves at the same level — much more regularized than LightGBM's 64 unbalanced leaves.

### Pitfall 4: Using `verbose=True` in Production

By default CatBoost prints a training log that's very verbose. Always set `verbose=0` (silence) or `verbose=N` (print every N iterations).

### Pitfall 5: Ignoring Class Imbalance

```python
# For imbalanced binary classification, use scale_pos_weight
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
model = CatBoostClassifier(
    class_weights=[1, neg/pos],   # weight minority class
    ...
)
# OR use auto_class_weights='Balanced'
model = CatBoostClassifier(auto_class_weights='Balanced', ...)
```

### Pitfall 6: Cross-Validation the Right Way

Always do CV **without** fitting any encoders on the full dataset first — CatBoost handles this internally, so just use `catboost.cv()` or sklearn's `cross_val_score` with the raw data.

## 12. Interview Q&A

---

**Q1: What is CatBoost and what makes it different from XGBoost and LightGBM?**

> CatBoost is a gradient-boosting library by Yandex with two unique features: (1) **native categorical handling** via Ordered Target Encoding — you can pass raw strings without any preprocessing, and (2) **Ordered Boosting** — an algorithm that computes each sample's residual using a model that has never seen that sample, eliminating prediction shift (a subtle form of training bias). XGBoost requires you to encode all categoricals manually; LightGBM can use integer-encoded categoricals but still requires encoding; CatBoost is the only library that handles raw string categoricals out of the box.

---

**Q2: What is ordered boosting? Why does it matter?**

> In standard gradient boosting, when computing residuals for sample i, the model that predicts for i was **trained on data that includes sample i**. This means the model has seen i's label, making residuals underestimated (the model is overconfident) — this is called **prediction shift**. Ordered boosting fixes this: it maintains multiple random orderings of the training data and, for each sample i, computes its residual using a model trained **only on samples that appeared before i** in that ordering. This produces unbiased residuals and better generalization, especially on small datasets.

---

**Q3: How does CatBoost handle categorical features internally (Ordered Target Encoding)?**

> For each categorical column, CatBoost replaces each category value with a **running target statistic** computed using a random permutation of the training data. For sample i with category C, the encoding is: `(sum of targets for all j < i where category_j == C + prior) / (count + 1)`. The prior is the global mean target. Since sample i's own label is never used when computing i's encoding, there's no leakage. Multiple random permutations are used across different trees to reduce variance.

---

**Q4: When would you choose CatBoost over LightGBM?**

> **Choose CatBoost when:**
> - Your dataset has many high-cardinality categorical features (city names, neighborhoods, product SKUs)
> - You want good out-of-the-box performance with minimal hyperparameter tuning
> - You need SHAP values and explainability (CatBoost has built-in SHAP)
> - Your training data is relatively small (ordered boosting helps more on small datasets)
> 
> **Choose LightGBM when:**
> - Speed is critical (LightGBM is typically fastest)
> - You have very large datasets (millions of rows)
> - Features are already numeric or you're comfortable with preprocessing

---

**Q5: What does `depth` mean in CatBoost, and how does it differ from `num_leaves` in LightGBM?**

> CatBoost uses **symmetric trees** by default: at each depth level, every node splits on the **same feature and same threshold**. This means a depth-6 tree has exactly 2^6 = 64 leaves, all at the same depth — like a complete binary tree. This is highly regularized and fast for prediction (inference is just 6 comparisons).
> 
> LightGBM uses **leaf-wise growth**: it always splits the leaf with the highest gain, regardless of depth. A `num_leaves=64` LightGBM tree can be very unbalanced — some branches go deep (high complexity) while others stay shallow. This is more flexible but less regularized.
> 
> Set `grow_policy='Lossguide'` to make CatBoost behave like LightGBM's leaf-wise approach.

---

**Q6: What is the Pool object in CatBoost?**

> `Pool` is CatBoost's native data container (analogous to XGBoost's `DMatrix` or LightGBM's `Dataset`). It preprocesses data once (quantizing numeric features, encoding categoricals) and stores it in an optimized format. Benefits: (1) avoids redundant preprocessing on repeated `.fit()` calls, (2) lets you specify column names, sample weights, and baseline predictions, (3) required for some advanced CatBoost methods like `get_feature_importance(type='ShapValues', data=pool)`.

---

**Q7: How do you handle class imbalance in CatBoost?**

> Two main approaches:
> 1. `auto_class_weights='Balanced'` — automatically weights classes inversely proportional to their frequency
> 2. `class_weights=[1, w]` — manually specify weights (e.g., `[1, neg_count/pos_count]`)
> 
> You can also use `scale_pos_weight` for binary classification. For extreme imbalance (1:100+), combine class weighting with threshold adjustment at inference time.

---

**Q8: How do you interpret CatBoost feature importance?**

> CatBoost offers three types:
> - **PredictionValuesChange**: average change in predicted values when a feature is removed (fast, global)
> - **LossFunctionChange**: average change in loss when a feature is removed (slower, more accurate)
> - **ShapValues**: per-sample additive decomposition of the prediction into feature contributions (most detailed, works for both local and global explanations)
> 
> SHAP values are the gold standard for interpretability: `shap_value[i, j]` tells you exactly how much feature j pushed sample i's prediction above or below the baseline.

## 13. Resources

### Official
- **CatBoost Documentation:** https://catboost.ai/en/docs/
- **GitHub Repository:** https://github.com/catboost/catboost
- **Original Research Paper (2017):** https://arxiv.org/abs/1706.09516
- **Ordered Target Encoding Paper:** https://arxiv.org/abs/1706.09516 (Section 3.2)

### Tutorials
- **CatBoost Tutorial Notebooks:** https://github.com/catboost/tutorials
- **Kaggle CatBoost Tutorial:** https://www.kaggle.com/code/dansbecker/catboost

### Videos
- **CatBoost explained by Yandex researchers:** https://www.youtube.com/watch?v=s3VmuVPfu0s
- **CatBoost vs XGBoost vs LightGBM (StatQuest style):** https://www.youtube.com/results?search_query=catboost+explained

### Comparison Studies
- **Why Does XGBoost Win? (Kaggle competitions analysis):** https://arxiv.org/abs/1602.07360
- **Benchmarking Gradient Boosting Libraries:** https://github.com/catboost/benchmarks

## 14. Summary & What's Next

### What You Learned

| Concept | Key Takeaway |
|---|---|
| **Native categoricals** | Pass raw strings directly via `cat_features` — no encoding required |
| **Ordered Target Encoding** | Encodes categories using only past samples in a random permutation — prevents leakage |
| **Ordered Boosting** | Computes residuals using models that haven't seen the current sample — prevents prediction shift |
| **Symmetric trees** | All leaves at same depth; depth-6 = 64 leaves; more regularized than LightGBM |
| **Pool object** | Pre-processes data once; use when training multiple models or doing cross-validation |
| **Early stopping** | Set `early_stopping_rounds` and always pass `eval_set` to find optimal iterations |
| **SHAP values** | Built-in support — explain any prediction in terms of feature contributions |
| **Class imbalance** | Use `auto_class_weights='Balanced'` or manual `class_weights` |

### The Gradient Boosting Decision Tree

```
Many high-cardinality categoricals?
        ↓ Yes                      ↓ No
   → CatBoost             Need maximum speed?
                               ↓ Yes              ↓ No
                          → LightGBM         → XGBoost or
                                               LightGBM
```

### What's Next

You've now completed **Classical Machine Learning** — from Scikit-Learn's foundational API, through XGBoost's gradient boosting intuition, LightGBM's leaf-wise speed, to CatBoost's categorical mastery.

**Up next: Deep Learning** — where you'll move from tree-based models to neural networks:
- **PyTorch** — the most popular deep learning framework for research and production
- **TensorFlow** — Google's production-grade deep learning platform
- **Keras** — high-level API that runs on top of TensorFlow
- **JAX** — NumPy-like library with automatic differentiation and JIT compilation